# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [1]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [2]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [3]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[-0.1746, -0.0776, -0.1062,  0.0700, -0.0220, -0.0005,  0.0152,  0.0012],
        [ 0.1644,  0.1204,  0.2139, -0.0301,  0.0417, -0.0905,  0.0386,  0.0441]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [4]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cpu
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [5]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [6]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[-0.1070,  0.0977],
        [ 0.1664, -0.1714],
        [ 0.1664, -0.1714],
        [ 0.9316, -0.6553],
        [ 1.3668, -0.4172]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[-0.0274, -0.1770],
        [-0.3446, -0.4921],
        [ 0.3285, -0.2925],
        [ 0.4231,  0.2323],
        [ 0.4807, -0.2101]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [8]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[-0.0066,  0.0617],
        [-0.3357, -0.0376],
        [ 0.3538,  0.3147],
        [ 0.2312, -0.6330],
        [ 0.6156, -0.3048]])
Output inference_mode method:
 tensor([[-0.0066,  0.0617],
        [-0.3357, -0.0376],
        [ 0.3538,  0.3147],
        [ 0.2312, -0.6330],
        [ 0.6156, -0.3048]])
Output no_grad context manager:
 tensor([[-0.0066,  0.0617],
        [-0.3357, -0.0376],
        [ 0.3538,  0.3147],
        [ 0.2312, -0.6330],
        [ 0.6156, -0.3048]])
Output inference_mode context manager:
 tensor([[-0.0066,  0.0617],
        [-0.3357, -0.0376],
        [ 0.3538,  0.3147],
        [ 0.2312, -0.6330],
        [ 0.6156, -0.3048]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [ ]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [9]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [10]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [11]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [12]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])



## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LSTMBlock(nn.Module):
    def __init__(self, input_size: int, hidden_size: int) -> None:
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        self.gate = nn.Linear(input_size + hidden_size, 4 * hidden_size, bias=True)
        
    def forward(self, x: torch.Tensor, initial_states=None) -> torch.Tensor:
        seq_len, batch_size, _ = x.shape
        
        # Initialize states
        if initial_states is None:
            h = torch.zeros(batch_size, self.hidden_size, device=x.device)
            c = torch.zeros(batch_size, self.hidden_size, device=x.device)
        else:
            h, c = initial_states
        
        outputs = []
        
        for t in range(seq_len):
            combined = torch.cat((x[t], h), dim=1)  # Shape: (batch_size, input_size + hidden_size)
            
            gates = self.gate(combined) 
            
            f_gate, i_gate, c_gate, o_gate = gates.chunk(4, dim=1)
            
            f = torch.sigmoid(f_gate)
            i = torch.sigmoid(i_gate)  
            c_tilde = torch.tanh(c_gate) 
            o = torch.sigmoid(o_gate)  
            
            c = f * c + i * c_tilde  
            h = o * torch.tanh(c) 
            
            outputs.append(h)
        output = torch.stack(outputs, dim=0)  # Shape: (seq_len, batch_size, hidden_size)
        
        return output

### Inception

![inception](assets/inception.png)

### SE

![se](assets/SqueezeAndExcite.png)

### Selective Kernel

![selective](assets/SelectiveKernel.png)


### PatchMerger

![patchmerger](assets/PatchMerger.png)


## Homework

2 задания:
1. Реализуйте требуемый в заголовке блок (максмсум 0.8 балов).

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [14]:
class ResidualBlock(nn.Module):
    def __init__(self, input_size: int):
        super().__init__()
        self.linear1 = nn.Linear(input_size, input_size)
        self.linear2 = nn.Linear(input_size, input_size)
        self.act = nn.ReLU()


    def forward(self, x : torch.Tensor):
        residual = x
        x = self.act(self.linear1(x))
        x = self.linear2(x)
        x += residual
        x = self.act(x)
        return x

## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [15]:
class SeparableConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int | tuple[int, int] = 3,
        stride: int | tuple[int, int] = 1,
        padding: int | tuple[int, int] | str = 1,
        dilation: int | tuple[int, int] = 1,
        bias: bool = False
    ):
        super().__init__()

        self.depthwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            groups=in_channels, #diffrent kernel for each input channel
            bias=bias
        )

        self.pointwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            bias=bias,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

In [ ]:
x = torch.randn(2, 4, 16) # (B, L, d)
conv = SeparableConv2d(in_channels=16, out_channels=8, kernel_size=3)
out = conv(x.permute(0, 2, 1).unsqueeze(-1))  # -> (B, C=16, H=4, W=1)
print("Output shape:", out.shape)

Output shape: torch.Size([2, 8, 4, 1])


## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [16]:
from typing import Optional
import torch
from torch import nn
import numpy as np

class VanillaAttention(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.W_align = nn.Linear(d, d)
        self.W_value = nn.Linear(d, d)
        self.W_query = nn.Linear(d, d)
        self.tanh = nn.Tanh()

    def forward(self, query: torch.Tensor, key: torch.Tensor):
        q = self.W_align(query) # (B, d)
        score = (key * q.unsqueeze(1)).sum(-1) # (B, L, d) * (B, 1, d) -> (B, L)
        att = torch.softmax(score, dim=-1) # (B, L)
        context = (att.unsqueeze(-1) * key).sum(1) # (B, L, 1) * (B, L, d) -> (B, d)
        out = self.tanh(self.W_query(query) + self.W_value(context)) # (B, d) + (B, d) -> (B, d)
        return out


## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [17]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.scale = d ** 0.5

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor):
        score = (query @ key.transpose(-2, -1)) / self.scale
        att = torch.softmax(score, dim=-1)
        out = att @ value
        return out

In [18]:
att = ScaledDotProductAttention(d=16)
B, L, d = 2, 4, 16
query = torch.randn(B, L, d, requires_grad=True)
key = torch.randn(B, L, d, requires_grad=True)
value = torch.randn(B, L, d, requires_grad=True)

# Forward pass
out = att(query, key, value)
print("Output shape:", out.shape)

Output shape: torch.Size([2, 4, 16])


## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        self.d = d
        self.head_dim = d // num_heads
        assert d % num_heads == 0, "d must be divisible by num_heads"

        self.q_proj = nn.Linear(d, d)
        self.k_proj = nn.Linear(d, d)
        self.v_proj = nn.Linear(d, d)
        self.out = nn.Linear(d, d)
        self.att = ScaledDotProductAttention(self.head_dim)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor):
        B, Lq, d = query.shape
        Bk, Lk, d = key.shape
        Bv, Lv, d = value.shape
        assert B == Bk == Bv
        Q = self.q_proj(query).view(B, Lq, self.num_heads, self.head_dim).transpose(1, 2) # (B, num_heads, Lq, head_dim)
        K = self.k_proj(key).view(B, Lk, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(value).view(B, Lv, self.num_heads, self.head_dim).transpose(1, 2)

        out = self.att(Q, K, V)

        out = out.transpose(1, 2).contiguous().view(B, L, self.d)
        out = self.out(out)
        return out

In [20]:
att = MultiHeadAttention(d=16, num_heads=4)
B, L, d = 2, 4, 16
query = torch.randn(B, L, d, requires_grad=True)
key = torch.randn(B, L, d, requires_grad=True)
value = torch.randn(B, L, d, requires_grad=True)

# Forward pass
out = att(query, key, value)
print("Output shape:", out.shape)

Output shape: torch.Size([2, 4, 16])


## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [21]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d: int, num_heads: int, dim_feedforward: int = 2048, dropout: float = 0.1 ):
        super().__init__()
        self.self_attn = MultiHeadAttention(d, num_heads)
        self.linear1 = nn.Linear(d, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d)
        self.mlp = nn.Sequential(
            nn.Linear(d, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d),
            nn.Dropout(dropout)

        )
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.norm1(x)
        x2 = self.self_attn(x1, x1, x1)
        x3 = x + x2
        x = self.norm2(x3)

        x = self.mlp(x)
        x = x + x3
        return x


In [22]:
x = torch.randn(2, 4, 16) # (B, L, d)
encoder_layer = TransformerEncoderLayer(d=16, num_heads=4)
out = encoder_layer(x)
print("Output shape:", out.shape)

Output shape: torch.Size([2, 4, 16])


## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [23]:
class MLPMixerBlock(nn.Module):
    def __init__(self, d: int, seq_len: int, hidden_dim: int, drop: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)

        self.token_mixing = nn.Sequential(
            nn.Linear(seq_len, hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden_dim, seq_len),
            nn.Dropout(drop),
        )
        self.channel_mixing = nn.Sequential(
            nn.Linear(d, hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden_dim, d),
            nn.Dropout(drop),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #x: (B, seq_len, d)
        y = self.ln1(x)
        y = y.transpose(1, 2)              # (B, d, seq_len)
        y = self.token_mixing(y)           # (B, d, seq_len)
        y = y.transpose(1, 2)              # (B, seq_len, d)
        x = x + y

        y = self.ln2(x)
        y = self.channel_mixing(y)         # (B, seq_len, d)
        x = x + y

        return x

In [24]:
x = torch.randn(2, 4, 16) # (B, L, d)
model = MLPMixerBlock(d=16, seq_len=4, hidden_dim=64)
out = model(x)
print("Output shape:", out.shape)

Output shape: torch.Size([2, 4, 16])


## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [38]:
class ConvMixer(nn.Module):

    def __init__(self, in_channels: int, kernel_size: int, num_blocks: int):
        super().__init__()
        self.deepwise = SeparableConv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2
        )   

        self.pointwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=1
        )
        self.act = nn.ReLU()
        self.batchnorm1 = nn.BatchNorm2d(in_channels)       
        self.batchnorm2 = nn.BatchNorm2d(in_channels)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.deepwise(x)
        x = self.act(x)
        x = self.batchnorm1(x)

        x = x + residual
        x = self.pointwise(x)
        x = self.act(x)
        x = self.batchnorm2(x)
        return x


In [40]:
x = torch.randn(2, 4, 16) # (B, L, d)
x = x.permute(0, 2, 1).unsqueeze(-1)  # -> (B, C=16, H=4, W=1)
model = ConvMixer(in_channels=16, kernel_size=3, num_blocks=2)
out = model(x)
print("Output shape:", out.shape)

Output shape: torch.Size([2, 16, 4, 1])


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ: Идея методов - замешать/миксовать информацию между каналами/токенами. Не все данные требуют  Multihead Attention, например, данные с локальными зависимостями могут быть хорошо обработаны и ConvMixer.
Общая формула, полагаю, это 
н = x + H(x) (см, например, https://arxiv.org/pdf/1512.03385)

1.  Multihead Attention самый вычислительно затратный блок (-)
2. В Multihead Attention может выбирать “куда смотреть” динамически (+)
3. Multihead Attention универсален (+)
4. MLPMixer часто линейный по L (+)
5. MLPMixer по токенам не контент-зависим (-)
6. MLPMixer зависит от длины последовательности (-)
7. ConvMixer работает  с локальными зависимостями (+ и -)